In [ ]:
import hostphot
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from hostphot.cutouts import download_images
from hostphot.coadd import coadd_images
from hostphot.image_masking import create_mask
from hostphot.utils import plot_fits, get_survey_filters
import hostphot.local_photometry as lp
import hostphot.global_photometry as gp
from hostphot.cutouts import set_HST_image
import pickle
import pandas as pd
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import shutil
import pickle
from astropy.cosmology import WMAP9 as cosmo
from astropy.io import fits
from astropy.wcs import WCS
import aplpy
from hostphot.cutouts import set_HST_image
import hostphot.local_photometry as lp

## Converting coordinates & photometry

In [ ]:
# Function to convert RA and DEC coordinates from hms and dms format to decimal degrees
def hms_dms_to_degrees(ra_hms, dec_dms):
    # Split the parts of the RA and DEC coordinates
    ra_h, ra_m, ra_s = map(float, ra_hms.split(':'))
    dec_d, dec_m, dec_s = map(float, dec_dms.split(':'))

    # Convert RA to decimal degrees
    ra_degrees = ra_h * 15 + (ra_m / 60) * 15 + (ra_s / 3600) * 15

    # Determine the sign of DEC
    sign = -1 if str(dec_d).startswith('-') else 1

    # Convert DEC to decimal degrees with the correct sign
    dec_degrees = sign * (abs(dec_d) + (dec_m / 60) + (dec_s / 3600))

    return ra_degrees, dec_degrees

# Load data from the updated CSV file
input_file = 'Prova coordenades supernoves - Sheet4.csv'
try:
    data = pd.read_csv(input_file)
    print(f"Data successfully loaded from '{input_file}'")
except FileNotFoundError:
    print(f"The file '{input_file}' does not exist. Check the name and path.")
    exit()

# Verify the columns present in the updated file
print("Columns present in the updated file:", data.columns.tolist())

# Check to ensure that all required columns exist
required_columns = ['Supernova', 'RA(H:M:S)', 'DEC(D:M:S)', 'Redshift z']
missing_columns = [col for col in required_columns if col not in data.columns]
if missing_columns:
    print(f"The following columns are missing in the file: {missing_columns}")
    exit()

# Create new columns for the transformed coordinates
data['RA(deg)'], data['DEC(deg)'] = zip(*data.apply(
    lambda row: hms_dms_to_degrees(
        row['RA(H:M:S)'], row['DEC(D:M:S)']
    ), axis=1
))

# Rename the RA and DEC columns to the desired units
data = data.rename(columns={
    'RA(deg)': 'RA(DEGREES)',
    'DEC(deg)': 'DEC(DEGREES)'
})

# Select the columns we want to save in the final file
output_data = data[['Supernova', 'RA(DEGREES)', 'DEC(DEGREES)', 'Redshift z']]

# Save the transformed data to a new CSV file
output_file = 'supernovas_transformed.csv'
output_data.to_csv(output_file, index=False)

pd.set_option('display.max_rows', None)  # Show all rows
print(f"Transformed data successfully saved to '{output_file}'")
print("Full content of the final file:")
print(output_data)

# Function to generate a plot with circles based on \sigma
def plot_supernova_custom_sigma(file, ra, dec, supernova_name, sigmas, output_name=None):
    try:
        with fits.open(file) as hdul:
            for hdu in hdul:
                if hdu.data is not None:
                    data = hdu.data
                    header = hdu.header
                    break
            else:
                print(f"No valid data found in FITS file: {file}")
                return

        # Configure WCS
        wcs = WCS(header)
        if not wcs.is_celestial:
            print(f"The FITS file '{file}' does not contain a valid celestial WCS system.")
            return

        # Transform coordinates
        try:
            x, y = wcs.world_to_pixel_values(ra, dec)
            print(f"Converting RA: {ra}, DEC: {dec} -> X: {x}, Y: {y}")  # Debugging to verify conversion
        except Exception as e:
            print(f"Error transforming coordinates with WCS: {e}")
            return

        # WCS diagnostics
        ny, nx = data.shape
        print(f"Image size: {nx}x{ny}")
        
        # Check that the coordinates are within the image
        if not (0 <= x < nx and 0 <= y < ny):
            print(f"The coordinates {ra}, {dec} of {supernova_name} fall outside the FITS boundaries.")
            return

        # Set the maximum acceptable radius in pixels to avoid exceeding the image
        max_radius_pixels = min(nx, ny) / 2  # Maximum radius based on half the image size

        fig = plt.figure(figsize=(6, 6))
        f = aplpy.FITSFigure(file, figure=fig)
        f.show_colorscale(cmap='viridis', stretch='linear')
        f.axis_labels.hide()
        f.tick_labels.hide()
        f.ticks.hide()

        # Draw circles with an adjusted radius
        for sigma in sigmas:
            # Convert sigma radius to pixels
            radius_in_pixels = sigma / 3600 * (nx / 3600)  # Convert to pixels, based on angular resolution

            # Limit the radius so it does not exceed the maximum allowed
            radius_in_pixels = min(radius_in_pixels, max_radius_pixels)

            f.show_circles(ra, dec, radius_in_pixels, edgecolor='red', lw=2)

        fixed_radius_arcsec = 3.0
        f.recenter(ra, dec, radius=fixed_radius_arcsec / 3600)

        plt.title(supernova_name, fontsize=14, color='black', pad=20)
        f.add_scalebar(1 / 3600)
        f.scalebar.set_label('1 arcsec')
        f.scalebar.set_color('white')
        f.scalebar.set_linewidth(2)
        f.scalebar.set_font(size=12, weight='bold')

        if output_name:
            os.makedirs(os.path.dirname(output_name), exist_ok=True)
            plt.savefig(output_name, dpi=300, bbox_inches='tight')
        plt.show()

    except Exception as e:
        print(f"Error processing {supernova_name}: {e}")

# Directory of FITS files
fits_directory = '/media/albert/TOSHIBA EXT/MAST_2024-11-20T13_24_24.714Z/MAST_2024-11-20T13_24_24.714Z/HST'
if not os.path.exists(fits_directory):
    raise FileNotFoundError(f"Directory {fits_directory} not found. Check that the external drive is mounted.")

# Read the transformed data
data = pd.read_csv('supernovas_transformed.csv')
results_dict = {}

for _, row in data.iterrows():
    name = row['Supernova']
    ra = row['RA(DEGREES)']
    dec = row['DEC(DEGREES)']
    z = row['Redshift z']

    survey = 'HST'
    filt = 'WFC3_UVIS_F275W'
    image_files = glob.glob(os.path.join(fits_directory, f'{name.lower()}_drc.fits'))

    if not image_files:
        print(f"No FITS files found for {name} in directory {fits_directory}.")
        continue

    kpc_per_arcsec = cosmo.kpc_proper_per_arcmin(z).value / 60.0
    ap_radii = 1

    for file in image_files:
        set_HST_image(file, filt, name)

        try:
            results = lp.multi_band_phot(
                name, ra, dec, z,
                filters=filt, survey=survey, ap_radii=ap_radii,
                correct_extinction=False, save_plots=True,
                raise_exception=True, use_mask=False
            )

            sigma_1 = results[f'{filt}_1_err']
            sigmas = [sigma_1, 2 * sigma_1, 3 * sigma_1]
            results_dict[file] = results

            output_name = f"plots/{name}_plot.png"
            plot_supernova_custom_sigma(file, ra, dec, name, sigmas=sigmas, output_name=output_name)

            supernova_dir = os.path.join('images', name)
            if os.path.exists(supernova_dir):
                shutil.rmtree(supernova_dir)
                print(f"Folder {supernova_dir} deleted.")

        except Exception as e:
            print(f"Error processing {file} for {name}: {e}")

# Save results
output_pickle = 'results.pkl'
with open(output_pickle, 'wb') as f:
    pickle.dump(results_dict, f)

print(f"Results saved to {output_pickle}")


## Results

In [ ]:
import pickle

# Load the pickle file
output_pickle = 'results.pkl'

try:
    with open(output_pickle, 'rb') as f:
        results_dict = pickle.load(f)
    print("Results loaded successfully.")
except FileNotFoundError:
    print(f"File {output_pickle} not found. Check the name and location.")
    exit()

# Display the results
if results_dict:
    for file, result in results_dict.items():
        print(f"Results for file: {file}")
        print(result)
        print("-" * 80)
else:
    print("The results file is empty.")


In [ ]:
import pickle
import pandas as pd

# Load the pickle file
pickle_file = 'results.pkl'
with open(pickle_file, 'rb') as f:
    results = pickle.load(f)

# Create a list to store the results
data = []

# Iterate through the results to extract the necessary data
for file, result in results.items():
    try:
        name = result['name']
        ra = result['ra']
        dec = result['dec']
        redshift = result['redshift']
        WFC3_UVIS_F275W_1 = result.get('WFC3_UVIS_F275W_1', None)
        WFC3_UVIS_F275W_1_err = result.get('WFC3_UVIS_F275W_1_err', None)
        WFC3_UVIS_F275W_1_flux = result.get('WFC3_UVIS_F275W_1_flux', None)
        WFC3_UVIS_F275W_1_flux_err = result.get('WFC3_UVIS_F275W_1_flux_err', None)

        # Add the data to the list
        data.append({
            'name': name,
            'ra': ra,
            'dec': dec,
            'redshift': redshift,
            'WFC3_UVIS_F275W_1': WFC3_UVIS_F275W_1,
            'WFC3_UVIS_F275W_1_err': WFC3_UVIS_F275W_1_err,
            'WFC3_UVIS_F275W_1_flux': WFC3_UVIS_F275W_1_flux,
            'WFC3_UVIS_F275W_1_flux_err': WFC3_UVIS_F275W_1_flux_err
        })
    except KeyError as e:
        print(f"Missing data for file {file}: {e}")

# Create a DataFrame with the results
df = pd.DataFrame(data)

# Display the formatted table
print(df.to_string(index=False))

print("Table successfully saved in LaTeX format: 'supernova_table.tex'")


## Distribution

In [ ]:
import pickle
import matplotlib.pyplot as plt
import numpy as np

# Load the file results.pkl
pickle_file_path = 'results.pkl'

with open(pickle_file_path, 'rb') as file:
    results_data = pickle.load(file)

# Filter valid magnitudes (exclude NaN or None values)
valid_magnitudes = []

for result in results_data.values():
    mag_key = 'WFC3_UVIS_F275W_1'
    mag = result.get(mag_key, None)
    
    # Check that the magnitude value is valid
    if mag is not None and not np.isnan(mag):
        valid_magnitudes.append(mag)

# Generate the histogram of magnitudes
plt.figure(figsize=(10, 6))
plt.hist(valid_magnitudes, bins=20, color='skyblue', edgecolor='black', alpha=0.7)  # Histogram with 20 bins

plt.xlabel('Magnitude', fontsize=12)
plt.ylabel('Number', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)  # Horizontal grid lines for better visualization
plt.tight_layout()

# Show the plot
plt.show()